# [실습용] 하이브리드 검색 & 리랭킹 기초 실습 — 의미 + 키워드 + 재정렬

## 👋 이 노트북을 보는 법

- ✏️ **실습용 빈칸 노트북입니다.** 코드 속 `___` 를 채운 뒤 실행하세요. (Shift+Enter)
- ✏️ 각 셀 위의 **💡 힌트**를 참고하세요. 빈칸을 채우지 않고 실행하면 오류가 납니다.
- ✏️ 셀은 **위에서부터 순서대로** 완성해야 합니다. (뒤 셀이 앞 셀의 함수를 사용)
- 🟢 **외부 파일이 필요 없습니다.** 샘플 문서를 코드로 정의합니다.
- 🟢 **API 키가 필요 없습니다.** 작은 모델만 자동 다운로드됩니다.
- 각 코드 셀 아래 **"이 셀이 한 일"** 로 결과가 맞는지 확인하세요.

## 오늘 배울 것 (딱 한 문장)

> 의미 검색은 **품번·코드 같은 정확한 단어에 약하다.**
> **키워드 검색을 합치고(하이브리드) → 리랭커로 재정렬**해 정확도를 끌어올린다.

| 단계 | 내용 |
|------|------|
| 1️⃣ 이론 | 의미 검색의 빈틈 → 하이브리드 → 리랭킹 (코드 없음) |
| 2️⃣ 기본 | 의미 검색 vs 키워드 검색(BM25) 비교 |
| 3️⃣ 응용 | RRF로 두 검색을 융합 (하이브리드) |
| 4️⃣ 데이터 활용 | 리랭커로 재정렬 · 전체 파이프라인 |


# 1️⃣ 단계 — 이론 (읽기만 하세요, 코드 없음)

## 의미 검색은 "뜻이 비슷한 것"을 잘 찾는다

```
 🧑 "포인트 언제 사라지나요?"
        │
        ▼ 의미 임베딩
 "적립금은 1년간 유효..." 청크를 찾음 ✅
   (글자는 다른데 뜻이 가까워서)
```

## 그런데 — 정확한 단어엔 약하다

```
 🧑 "모델 ABC-1234 소비전력"
        │
        ▼ 의미 임베딩
 ABC-1234 와 의미가 비슷한
 ABC-1233 · ABC-1200 · XYZ-1234 도 섞여 옴 ❌
        ▲
   품번은 '의미'가 아니라 '정확히 그 글자'가 중요한데!
```

## 해결 — 키워드 검색을 합치고, 재정렬한다

```
 ① 하이브리드   의미 검색(Dense) + 키워드 검색(BM25)
               → 점수 대신 '순위'로 융합 (RRF)
               → 양쪽에서 두루 상위인 문서를 위로

 ② 리랭킹       넓게 20~30개 뽑은 뒤,
               질문+문서를 함께 보는 정밀 채점기로
               진짜 관련 top 3~5만 남김
```

> 📌 핵심: **개념 질문은 의미 검색이 강하고, 정확한 토큰 질문은
> 키워드 검색이 강하다.** 둘은 서로의 약점을 메웁니다. 합치면 강해집니다.


# 2️⃣ 단계 — 의미 검색 vs 키워드 검색

### 셀 1 — 도구 설치

In [ ]:
# 셀 1 — 도구 설치
# !  → 터미널 명령 (Colab/Jupyter 전용)   -q → 조용히(로그 최소)
# sentence-transformers : 문장을 '의미 벡터'로 바꾸는 임베딩 모델 모음
# rank_bm25             : 키워드(단어 일치) 기반 검색 BM25 구현체
!pip install -q sentence-transformers rank_bm25


**이 셀이 한 일**: 실습 도구를 설치했습니다.
`sentence-transformers`(의미 검색용 임베딩) · `rank_bm25`(키워드 검색).


### 셀 2 — 샘플 문서 정의

품번이 비슷한 제품 문서 + 개념 안내 문서를 섞어 둡니다.
(의미 검색이 강한 질문 / 약한 질문을 둘 다 만들기 위해)


In [ ]:
# 셀 2 — 샘플 문서 정의 (외부 파일 없이 코드로 직접 만듦)
docs = [
    # 0~3번: 품번이 '거의 비슷한' 제품들 → 정확한 글자가 중요(의미검색이 약한 영역)
    "제품 ABC-1234 사양: 소비전력 45W, 무게 1.2kg, 보증기간 2년입니다.",
    "제품 ABC-1233 사양: 소비전력 40W, 무게 1.1kg, 보증기간 2년입니다.",
    "제품 ABC-1200 사양: 소비전력 30W, 무게 0.9kg, 보증기간 1년입니다.",
    "제품 XYZ-1234 사양: 소비전력 60W, 무게 1.5kg, 보증기간 3년입니다.",
    # 4~7번: 개념 안내 → 글자가 달라도 '뜻'만 가까우면 됨(의미검색이 강한 영역)
    "환불 규정: 구매 후 7일 이내 미개봉 상품에 한해 전액 환불이 가능합니다.",
    "교환 안내: 단순 변심 교환은 14일 이내 가능하며 배송비는 고객 부담입니다.",
    "적립금 안내: 적립된 적립금은 적립일로부터 1년간 사용 후 소멸됩니다.",
    "배송 정책: 평일 오후 2시 이전 결제된 주문은 당일 출고됩니다.",
]

# enumerate : 번호(i)와 문서(d)를 같이 받아 출력 → 몇 번 문서가 뭔지 눈으로 확인
for i, d in enumerate(docs):
    print(i, d)


**이 셀이 한 일**: 문서 8개를 정의했습니다.
0~3번은 **품번이 비슷한 제품**(정확한 단어가 중요),
4~7번은 **개념 안내**(뜻이 비슷하면 됨)입니다.


### 셀 3 — 의미 검색 (강점과 빈틈을 동시에 확인)

문장을 임베딩해 **뜻이 가까운** 문서를 찾습니다.
개념 질문엔 강하고, 정확한 품번 질문엔 약함을 봅니다.

> 💡 **힌트** (빈칸 4개)
> - 길이를 1로 정규화하는 옵션 값 → `True` / `False`?
> - 행렬곱 연산자 (1글자)
> - `np.argsort`는 오름차순 → 내림차순으로 만들려면?
> - 상위 `top_k`개만 자르는 슬라이싱

In [ ]:
# 셀 3 — 의미 검색 (강점과 빈틈을 동시에 확인)
from sentence_transformers import SentenceTransformer
import numpy as np                          # 벡터 계산용

# 다국어 소형 임베딩 모델 로드 (처음 실행 시 자동 다운로드, API 키 불필요)
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# 문서 8개를 한 번에 '의미 벡터'로 변환
#   normalize_embeddings : 길이를 1로 맞춤 → 내적이 곧 코사인 유사도가 됨
doc_embeddings = model.encode(docs, normalize_embeddings=___)   # TODO ①

def semantic_search(query, top_k=3):
    q = model.encode(query, normalize_embeddings=True)  # 질문도 같은 방식으로 벡터화
    sims = doc_embeddings ___ q   # TODO ② 행렬곱 → 각 문서와의 유사도 점수 배열
    ranked = np.argsort(___)   # TODO ③ 점수 높은 순(내림차순) 인덱스
    return [(int(i), float(sims[i])) for i in ___]   # TODO ④  # 상위 top_k개 (id, 점수)

# 두 종류 질문으로 비교: 개념 질문 vs 정확한 품번 질문
for query in ["포인트 언제 사라지나요?", "모델 ABC-1234 소비전력"]:
    print("🧑", query)
    for i, s in semantic_search(query):
        print(f"   {s:.3f}  {docs[i]}")     # {s:.3f} : 점수를 소수 셋째자리까지
    print()


**이 셀이 한 일**: 의미 검색을 돌렸습니다.

> ✅ "포인트 언제 사라지나요?" → 글자가 다른 **적립금** 문서를 잘 찾음 (강점).
> ❌ "ABC-1234 소비전력" → ABC-1233·ABC-1200·XYZ-1234 가 **섞여 옴** (빈틈).


### 셀 4 — 키워드 검색 BM25 (정확한 단어에 강함)

`BM25` 는 "질문의 단어가 문서에 실제로 있나"로 점수를 매깁니다.
의미 검색과 **정반대 강점**을 봅니다.

> 💡 **힌트** (빈칸 3개)
> - 문자열을 소문자로 바꾸는 메서드
> - 위에서 import 한 BM25 클래스 이름
> - 질문 토큰을 넣어 모든 문서의 점수를 받는 메서드: `get_______`

In [ ]:
# 셀 4 — 키워드 검색 BM25 (정확한 단어에 강함)
from rank_bm25 import BM25Okapi

def tokenize(text):
    # 대소문자 통일 후 공백으로 단어 분리
    #   "ABC-1234"는 하이픈을 안 쪼개 한 토큰으로 유지 → 품번 정확 매칭의 핵심
    return text.___().split()   # TODO ①

tokenized_docs = [tokenize(d) for d in docs]  # 모든 문서를 단어 목록으로 변환
bm25 = ___(tokenized_docs)   # TODO ②             # BM25 색인 생성(단어 빈도 기반)

def keyword_search(query, top_k=3):
    scores = bm25.___(tokenize(query))   # TODO ③ # 질문 단어가 각 문서에 얼마나 있나 점수화
    ranked = np.argsort(-scores)              # 점수 높은 순 정렬
    return [(int(i), float(scores[i])) for i in ranked[:top_k]]

for query in ["포인트 언제 사라지나요?", "모델 ABC-1234 소비전력"]:
    print("🧑", query)
    for i, s in keyword_search(query):
        print(f"   {s:.3f}  {docs[i]}")
    print()


**이 셀이 한 일**: 키워드 검색(BM25)을 돌렸습니다.

> ✅ "ABC-1234 소비전력" → **ABC-1234를 정확히** 1등으로 찾음 (의미 검색 빈틈을 메움).
> ❌ "포인트 언제 사라지나요?" → "포인트"라는 글자가 없는 적립금 문서를 못 찾음.
> 👉 의미 vs 키워드는 **서로 반대 강점** — 그래서 합칩니다.


# 3️⃣ 단계 — RRF로 두 검색을 융합 (하이브리드)

### 셀 5 — RRF 함수 직접 구현

두 검색은 **점수 척도가 다릅니다**(벡터 0~1, BM25 0~수십).
그냥 더하면 한쪽이 짓눌리므로 — **점수 대신 '순위'로 합칩니다.**

```
 RRF 점수 = Σ 1/(k + 순위)   → 양쪽에서 두루 상위인 문서가 올라감
```

> 💡 **힌트** (빈칸 3개)
> - 순위는 몇 등부터 시작하나요?
> - 위 RRF 공식을 코드로 옮기기 (`k`, `rank` 사용)
> - 점수 높은 순 정렬 → `reverse=` ?

In [ ]:
# 셀 5 — RRF 함수 직접 구현 (점수 대신 '순위'로 합치기)
# 의미검색(0~1)과 BM25(0~수십)는 점수 척도가 달라 그냥 더하면 한쪽이 짓눌림
# → 점수는 버리고 '몇 등이었나(순위)'만 사용해 공정하게 융합
def rrf(rank_lists, k=60):
    scores = {}                              # {문서id: 누적 RRF점수}
    for ranking in rank_lists:               # ranking = 순위대로 정렬된 문서 id 목록
        # enumerate(..., start=?) : 1등, 2등, 3등... 순위 번호 부여
        for rank, doc_id in enumerate(ranking, start=___):   # TODO ①
            # 핵심 공식: 상위(순위 작음)일수록 값이 커짐 → 점수 가산
            #   k=60 : 1~2등 차이를 너무 과하게 벌리지 않게 완충(현업 관례값)
            scores[doc_id] = scores.get(doc_id, 0.0) + ___   # TODO ②
    # 누적 점수가 높은 순으로 문서 id 정렬해 반환
    return sorted(scores, key=scores.get, reverse=___)   # TODO ③

# 이론 예시로 동작 확인 (실제 검색 대신 가짜 순위로 검증)
semantic_rank = ["D", "A", "B"]              # 의미검색 결과 순위
keyword_rank  = ["A", "D", "C"]              # 키워드검색 결과 순위
print("의미 순위 :", semantic_rank)
print("키워드순위:", keyword_rank)
# A: 의미 2등 + 키워드 1등 → 양쪽 모두 상위라 최종 1등이 됨
print("RRF 융합  :", rrf([semantic_rank, keyword_rank]))


**이 셀이 한 일**: RRF를 구현하고 작은 예시로 검증했습니다.

> 💡 A는 의미 2등 + 키워드 1등으로 **양쪽 모두 상위** → 최종 1등.
> 점수 스케일이 달라도 안전하고 구현이 단순해 현업 표준입니다.


### 셀 6 — 하이브리드 검색 (의미만 vs 하이브리드 비교)

의미 검색과 키워드 검색을 둘 다 돌려 RRF로 융합합니다.
정확한 품번 질문에서 **의미만** vs **하이브리드**가 어떻게 갈리는지 봅니다.

> 💡 **힌트** (빈칸 3개)
> - 셀 3·셀 4에서 만든 검색 함수 이름
> - 셀 5에서 만든 융합 함수 이름

In [ ]:
# 셀 6 — 하이브리드 검색 (의미만 vs 하이브리드 비교)
def hybrid_search(query, top_k=3, pool=5):
    # ① 의미·키워드 각각 pool개씩 넓게 후보를 뽑음 (점수는 버리고 id만)
    sem_ids = [i for i, _ in ___(query, pool)]   # TODO ① 의미 검색
    key_ids = [i for i, _ in ___(query, pool)]   # TODO ② 키워드 검색
    # ② 두 순위를 RRF로 융합 → 양쪽에서 두루 상위인 문서가 위로
    fused = ___([sem_ids, key_ids])   # TODO ③ 순위 융합
    return fused[:top_k]                     # ③ 융합 결과 상위 top_k만

query = "모델 ABC-1234 소비전력"             # 정확한 품번이 중요한 질문
print("🧑", query, "\n")

print("─── 의미 검색만 ───")
for i, s in semantic_search(query, 3):       # 비슷한 품번이 섞여 나옴(빈틈)
    print("  ", docs[i])

print("\n─── 하이브리드(의미+키워드) ───")
for i in hybrid_search(query, 3):            # 키워드가 ABC-1234를 끌어올려 보정
    print("  ", docs[i])


**이 셀이 한 일**: 같은 질문을 의미만 vs 하이브리드로 검색했습니다.

> ✅ 의미만 → 비슷한 품번이 섞임.
> 하이브리드 → **ABC-1234가 정확히 상위**로 올라옴.
> **검색기만 바꿔** 정확도가 달라집니다.


# 4️⃣ 단계 — 리랭커로 재정렬 · 전체 파이프라인

### 셀 7 — 리랭커로 정밀 재채점 (현업 치트키)

검색기는 '대충 비슷한 순서'로 줄 뿐, 정답이 1등이란 보장이 없습니다.
**리랭커(Cross-Encoder)** 는 질문+문서를 *함께* 넣어 정밀하게 다시 채점합니다.

```
 검색 임베딩(Bi-Encoder)   리랭커(Cross-Encoder)
 질문·문서 따로 임베딩  vs  질문+문서 '함께' 채점
 빠르지만 거침              느리지만 정밀
```

> 💡 **힌트** (빈칸 3개)
> - 리랭커에 넣는 쌍은 `(질문, 문서 내용)` → 문서 내용은 `docs[i]`
> - 점수를 예측하는 메서드 이름 (예측 = predict)

In [ ]:
# 셀 7 — 리랭커로 정밀 재채점 (현업 치트키)
from sentence_transformers import CrossEncoder

# Cross-Encoder : 질문과 문서를 '함께' 넣어 관련도를 정밀 채점 (느리지만 정확)
#   ↔ 검색 임베딩(Bi-Encoder): 따로 임베딩해 빠르지만 거침
reranker = CrossEncoder("cross-encoder/mmarco-mMiniLMv2-L12-H384-v1")

def rerank(query, doc_ids, top_k=3):
    pairs = [(___, ___) for i in doc_ids]   # TODO ①②  # (질문, 문서) 쌍으로 묶기
    scores = reranker.___(pairs)   # TODO ③            # 각 쌍의 관련도 정밀 점수
    order = np.argsort(-scores)                  # 점수 높은 순 정렬
    return [(doc_ids[o], float(scores[o])) for o in order[:top_k]]

query = "모델 ABC-1234 소비전력"
# ① 넓게: 하이브리드로 후보 6개를 일단 모음 (놓치지 않게 넉넉히)
candidates = hybrid_search(query, top_k=6, pool=8)
print("넓게 검색한 후보:", [docs[i][:16] for i in candidates], "\n")
# ② 정밀: 그 후보만 리랭커가 다시 정밀 채점해 진짜 top-3만 남김
print("리랭커가 정밀 재채점한 top-3:")
for i, s in rerank(query, candidates, 3):
    print(f"   {s:.3f}  {docs[i]}")


**이 셀이 한 일**: 넓게 모은 후보를 리랭커가 정밀 재채점했습니다.

> 💡 1차 검색은 넓게 거르고, 느리지만 정밀한 리랭커가 좁힙니다.
> 이 **"넓게 → 정밀" 2단계**가 검색 정확도를 가장 크게 끌어올립니다.


### 셀 8 — 전체 파이프라인: 넓게 → 융합 → 정밀

지금까지 만든 조각을 하나로 잇습니다:
**하이브리드로 넓게 → RRF 융합 → 리랭커로 top-3**.

> 💡 **힌트** (빈칸 2개)
> - ①②는 셀 6의 함수, ③은 셀 7의 함수

In [ ]:
# 셀 8 — 전체 파이프라인: 넓게 → 융합 → 정밀
# 지금까지 만든 조각을 한 함수로 연결한 '검색 파이프라인'
def rag_search(query, broad=8, final=3):
    # ①넓게 + ②융합 : 하이브리드(의미+키워드+RRF)로 broad개 후보 확보
    candidates = ___(query, top_k=broad, pool=broad)   # TODO ①
    # ③정밀 : 리랭커가 재채점해 최종 final개만 반환
    return ___(query, candidates, final)   # TODO ②

# 정확한 품번 질문 + 개념 질문 → 둘 다 잘 되는지 한 번에 검증
for query in ["모델 ABC-1234 소비전력", "포인트 언제 사라지나요?"]:
    print("🧑", query)
    for i, s in rag_search(query):
        print(f"   {s:.3f}  {docs[i]}")
    print()


**이 셀이 한 일**: 검색 파이프라인을 완성했습니다.

> ✅ 정확한 품번 질문(ABC-1234)도, 개념 질문(포인트)도 **둘 다** 정확히 답합니다.
> 검색 체인 전체를 바꾸지 않고 **검색 부분만** 이 구조로 교체하면 됩니다.


---
# 🎓 정리 — 적용한 이론

| 이론 | 이 실습에서 한 일 |
|------|-------------------|
| 의미 검색의 빈틈 | 정확한 품번에서 비슷한 코드가 섞임을 확인 |
| 키워드 검색(BM25) | 정확한 품번을 정확히 찾아 빈틈을 메움 |
| 하이브리드 | 의미+키워드를 둘 다 돌려 결과 융합 |
| RRF | 점수 대신 순위로 합쳐 양쪽 상위를 위로 |
| 리랭킹 | 넓게 검색 후 정밀 채점기로 top 압축 |
| 넓게 → 정밀 | 하이브리드 → 리랭커 2단계 파이프라인 |

**한 문장 요약**: 의미 검색은 정확한 단어에 약하다. 키워드(BM25)와
합치고(하이브리드+RRF), 리랭커로 정밀히 재정렬하면 검색 정확도가
가장 크게 오른다. 검색 부분만 교체하면 된다.

## 핵심 비교

```
                의미만        하이브리드      +리랭킹
 정확한 품번    섞임 ❌        정확 ✅         정답 top ✅
 개념 질문      정확 ✅        정확 ✅         정답 top ✅
```

## 🤔 더 해보기 (선택)

1. 셀 5의 `rrf` 의 `k` 값을 `10`, `100` 으로 바꿔 융합 순위가 어떻게
   달라지는지 관찰해 보세요.
2. 셀 6의 `query` 를 `"제50조의2"` 처럼 비슷한 번호가 섞이는 질문으로
   바꿔도 하이브리드가 정확히 잡는지 확인해 보세요. (docs에 조문 문서를 추가해도 좋습니다)
3. 셀 8의 `broad`(넓게 뽑는 개수)를 늘리면 리랭커가 더 좋은 후보를
   고를 수 있는지, 대신 느려지는지 비교해 보세요.
4. 검색이 정확해졌는데도 청크가 작게 잘려 *"절차 ②만 답하고 ①·③은 빠지는"*
   문제는 왜 별개일까요? "검색용 청크"와 "LLM 입력용 청크"를 다르게
   쓸 수 있을지 적어보세요.
